# 1. Treniranje neuronske mreže za prepoznavanje satelitskih snimki

U ovoj bilježnici učimo **konvolucijsku neuronsku mrežu** (CNN) da prepozna što se nalazi na satelitskoj snimci: šuma, rijeka, autocesta, naselje...

**Skup podataka EuroSAT** sadrži snimke satelita **Sentinel-2** (europski program Copernicus). Svaka snimka ima 64 × 64 piksela, a jedan piksel pokriva oko 10 × 10 m terena – cijela snimka prikazuje otprilike 640 × 640 m. Snimke su podijeljene u 10 razreda.

**Plan:**
1. provjeriti računalo i odabrati postavke,
2. učitati i pogledati podatke,
3. pripremiti podatke za učenje,
4. napraviti model,
5. trenirati model i spremiti najbolju verziju,
6. nacrtati kako je učenje napredovalo.

Ćelije pokreći redom, od vrha prema dnu (**Shift + Enter**). Kod koji je zajednički za sve tri bilježnice nalazi se u datoteci `eurosat_common.py` – sve što u kodu počinje s `ec.` dolazi odande.

## Korak 0: Instalacija paketa

PyTorch je već instaliran. Doinstaliramo `datasets` (preuzimanje skupa podataka s Hugging Facea) i `matplotlib` (crtanje grafova). Dovoljno je pokrenuti jednom.

In [ ]:
%pip install -q datasets matplotlib

## Korak 1: Uvoz biblioteka i provjera računala

Funkcija `ec.get_device()` provjerava imamo li grafičku karticu (GPU). Na grafičkoj kartici učenje je i do stotinu puta brže nego na procesoru (CPU).

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from matplotlib.ticker import MaxNLocator
from torch import nn
from torch.utils.data import DataLoader

import eurosat_common as ec

device = ec.get_device()

## Korak 2: Postavke (hiperparametri)

Ovim brojevima upravljamo učenjem. Kad sve proradi, slobodno ih mijenjaj i promatraj što se događa!

In [ ]:
EPOCHS = 20                # koliko puta model prođe kroz sve slike za treniranje
BATCH_SIZE = 128           # koliko slika model obradi odjednom (jedna "serija")
LEARNING_RATE = 0.001      # stopa učenja: koliko velike korake model radi dok uči
VALIDATION_FRACTION = 0.1  # udio slika za treniranje koje odvajamo za provjeru (10 %)
NUM_WORKERS = 2            # broj pomoćnih procesa koji pripremaju slike; ako dobiješ grešku "bus error" ili "shared memory", stavi 0
SEED = 42                  # "sjeme" slučajnosti: isti broj daje (gotovo) iste rezultate pri svakom pokretanju

torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True  # neka grafička kartica sama odabere najbrži način računanja

## Korak 3: Učitavanje podataka

Funkcija `load_dataset` preuzme skup s Hugging Facea (oko 100 MB, samo prvi put – poslije se učitava s diska).

Skup ima više dijelova: `train` za treniranje, `test` za završnu provjeru i 7 iskvarenih verzija testnog skupa (šum, zamućenje...) koje koristimo u 2. bilježnici.

In [ ]:
dataset = load_dataset(ec.DATASET_ID)
print(dataset)

Koliko ima slika svakog razreda? Ako bi nekog razreda bilo puno manje, model bi ga slabije naučio.

In [ ]:
train_full = dataset["train"]
counts = torch.bincount(torch.tensor(list(train_full["label"])), minlength=ec.NUM_CLASSES)

plt.figure(figsize=(10, 4))
plt.barh(ec.CLASS_NAMES, counts, color="#2a78d6")
plt.gca().invert_yaxis()
plt.xlabel("broj slika")
plt.title("Broj slika za treniranje po razredima")
plt.tight_layout()
plt.show()

Pogledajmo 20 nasumičnih slika. Bi li ti znao/znala prepoznati razred bez natpisa?

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(14, 12))
for ax, index in zip(axes.flat, torch.randperm(len(train_full))[:20].tolist()):
    example = train_full[index]
    ax.imshow(example["image"])
    ax.set_title(ec.CLASS_NAMES[example["label"]], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Korak 4: Skup za validaciju

Model ne smije vidjeti testne slike sve do samog kraja – inače ne bismo znali radi li dobro na **novim** slikama. Zato od slika za treniranje odvojimo 10 % u **validacijski skup**. Na njemu nakon svake epohe provjeravamo napredak i biramo najbolju verziju modela.

`stratify_by_column="label"` pazi da svaki razred bude jednako zastupljen u oba dijela.

In [ ]:
split = train_full.train_test_split(test_size=VALIDATION_FRACTION, seed=SEED, stratify_by_column="label")
train_data = split["train"]
val_data = split["test"]  # Hugging Face drugi dio zove "test", ali za nas je to validacijski skup

print("Slika za treniranje:", len(train_data))
print("Slika za validaciju:", len(val_data))

## Korak 5: Augmentacija – više različitih slika iz istih podataka

Snimka šume i dalje je šuma ako je zrcalimo ili joj malo promijenimo svjetlinu. Zato tijekom treniranja svaku sliku nasumično mijenjamo. Model tako u svakoj epohi vidi malo drukčije slike i uči opća pravila umjesto da slike pamti napamet. Pogledaj `train_transform` u datoteci `eurosat_common.py`.

Ispod je ista slika 8 puta provučena kroz augmentaciju:

In [ ]:
image = train_data[0]["image"]

fig, axes = plt.subplots(1, 9, figsize=(18, 2.5))
axes[0].imshow(image)
axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(ec.tensor_to_image(ec.train_transform(image)))
    ax.set_title("augmentirana")
for ax in axes:
    ax.axis("off")
plt.show()

## Korak 6: DataLoader – slike u serijama

Model ne uči sliku po sliku, nego u **serijama** (*batch*) od `BATCH_SIZE` slika – grafička kartica tako radi mnogo brže.

- `ec.EuroSATDataset` za svaku sliku vraća par *(slika kao tenzor brojeva, oznaka razreda)*.
- `DataLoader` slaže te parove u serije. `shuffle=True` u svakoj epohi promiješa redoslijed slika za treniranje.

In [ ]:
train_loader = DataLoader(
    ec.EuroSATDataset(train_data, ec.train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    ec.EuroSATDataset(val_data, ec.eval_transform),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)

images, labels = next(iter(train_loader))
print("Oblik serije slika:", tuple(images.shape), " -> [broj slika, boje (RGB), visina, širina]")
print("Oblik serije oznaka:", tuple(labels.shape))
print("Prvih 10 oznaka:", labels[:10].tolist())

## Korak 7: Model – konvolucijska neuronska mreža

Model `SimpleCNN` definiran je u `eurosat_common.py` i ima dva dijela:

- **`features`** – 4 konvolucijska bloka. Svaki blok traži uzorke na slici (rubove, teksture, oblike) i smanji sliku na pola: 64 → 32 → 16 → 8 → 4 piksela. Što smo dublje, piksela je manje, ali je više **mapa značajki** (32 → 64 → 128 → 256) koje opisuju sve složenije uzorke.
- **`classifier`** – od 256 značajki izračuna 10 brojeva, po jedan za svaki razred. Najveći broj je odgovor modela.

In [ ]:
model = ec.SimpleCNN().to(device)
print(model)

num_params = sum(p.numel() for p in model.parameters())
print(f"\nBroj parametara (brojeva koje model uči): {num_params:,}")

Provučemo kroz model seriju slika iz prethodnog koraka. Model još ništa nije naučio, pa su rezultati zasad slučajni.

In [ ]:
with torch.no_grad():
    outputs = model(images.to(device))

print("Ulaz:", tuple(images.shape), " -> izlaz:", tuple(outputs.shape))
print("10 rezultata za prvu sliku:", outputs[0].cpu().numpy().round(2))

## Korak 8: Funkcija gubitka, optimizator i stopa učenja

- **Funkcija gubitka** (`CrossEntropyLoss`) mjeri koliko model griješi. Što je gubitak manji, to bolje – učenje je zapravo smanjivanje gubitka.
- **Optimizator** (`Adam`) nakon svake serije malo promijeni parametre modela u smjeru u kojem se gubitak smanjuje.
- **Raspored stope učenja** (`CosineAnnealingLR`) polako smanjuje stopu učenja od `LEARNING_RATE` prema nuli: na početku model radi velike korake, a pri kraju sitne i precizne.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## Korak 9: Treniranje

Jedna **epoha** znači da je model jednom vidio sve slike za treniranje. Za svaku seriju slika:
1. model izračuna predviđanja,
2. izračunamo gubitak,
3. izračunamo kako svaki parametar utječe na gubitak (*backward* – povratno širenje),
4. optimizator popravi parametre.

Nakon svake epohe provjerimo model na validacijskom skupu. Kad je točnost dosad najbolja, model spremimo u datoteku (★).

`ec.mixed_precision(device)` uključuje **mješovitu preciznost**: grafička kartica računa s manje preciznim brojevima (bfloat16), što je brže, a za učenje dovoljno točno.

In [ ]:
def train_one_epoch(model, loader):
    """Jedna epoha treniranja. Vraća prosječni gubitak i točnost."""
    model.train()  # način rada za treniranje (npr. Dropout je uključen)
    total_loss, correct, seen = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        with ec.mixed_precision(device):
            outputs = model(images)          # 1. predviđanje
            loss = loss_fn(outputs, labels)  # 2. gubitak

        optimizer.zero_grad()  # obriši gradijente iz prethodne serije
        loss.backward()        # 3. povratno širenje
        optimizer.step()       # 4. popravak parametara

        total_loss += loss.item() * len(labels)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += len(labels)

    return total_loss / seen, correct / seen

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_epoch = 0.0, 0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, _, _ = ec.evaluate(model, val_loader, device)
    scheduler.step()  # smanji stopu učenja za sljedeću epohu

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    message = (f"Epoha {epoch:2d}/{EPOCHS} | "
               f"trening: gubitak {train_loss:.3f}, točnost {train_acc:6.1%} | "
               f"validacija: gubitak {val_loss:.3f}, točnost {val_acc:6.1%} | "
               f"{time.time() - start:4.1f} s")

    if val_acc > best_val_acc:
        best_val_acc, best_epoch = val_acc, epoch
        ec.save_model(model, ec.MODEL_PATH, epoch=epoch, val_accuracy=val_acc)
        message += "  ★ spremljen najbolji model"
    print(message)

## Korak 10: Grafovi učenja

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 4.5))

ax_loss.plot(epochs, history["train_loss"], "o-", color="#2a78d6", linewidth=2, label="trening")
ax_loss.plot(epochs, history["val_loss"], "o-", color="#eb6834", linewidth=2, label="validacija")
ax_loss.set_title("Gubitak (manje je bolje)")
ax_loss.set_xlabel("epoha")
ax_loss.legend()

ax_acc.plot(epochs, [a * 100 for a in history["train_acc"]], "o-", color="#2a78d6", linewidth=2, label="trening")
ax_acc.plot(epochs, [a * 100 for a in history["val_acc"]], "o-", color="#eb6834", linewidth=2, label="validacija")
ax_acc.axvline(best_epoch, color="gray", linestyle="--", label=f"najbolja epoha ({best_epoch})")
ax_acc.set_title("Točnost u % (više je bolje)")
ax_acc.set_xlabel("epoha")
ax_acc.legend()

for ax in (ax_loss, ax_acc):
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))  # na osi x samo cijeli brojevi epoha

plt.show()

**Kako čitati grafove?**

- Ako obje krivulje gubitka padaju, model uči.
- Ako gubitak na treniranju i dalje pada, a na validaciji počne rasti, model je počeo **pretjerano učiti** (*overfitting*): pamti slike za treniranje napamet umjesto da uči opća pravila. Zato spremamo model iz epohe s najboljom točnošću na validaciji, a ne onaj iz zadnje epohe.
- Točnost na treniranju može biti *niža* nego na validaciji jer model tijekom treniranja vidi augmentirane slike, a Dropout mu "isključuje" dio neurona – to mu otežava zadatak.

## Sažetak

In [ ]:
print(f"Najbolja točnost na validaciji: {best_val_acc:.1%} (epoha {best_epoch})")
print(f"Model je spremljen u datoteku: {ec.MODEL_PATH}")
print("Sljedeći korak: bilježnica 2_testiranje.ipynb")

### Za istraživanje

Promijeni jednu postavku u koraku 2, pokreni bilježnicu ponovno i usporedi grafove:

- `EPOCHS = 5` ili `EPOCHS = 40` – treba li modelu više vremena?
- `LEARNING_RATE = 0.01` ili `LEARNING_RATE = 0.0001` – što ako su koraci preveliki ili premali?
- `BATCH_SIZE = 32` ili `BATCH_SIZE = 512` – kako se mijenja brzina epohe i točnost? (Za velike serije postavi i `NUM_WORKERS = 0`, inače se može pojaviti greška "shared memory".)
- bez augmentacije: u koraku 6 zamijeni `ec.train_transform` s `ec.eval_transform` – pojavljuje li se pretjerano učenje?